# Fitting Normal Distribution to Data with Uncertainties

This notebook walks the user through a nominally very simple exercise; fitting a normal distribution to a set of data points (with errors).
Naively one might simply take the mean and standard deviation of the points which to first order is likely to be a good estimate.
However, since these points have associated uncertainties, the problem is not so straight forward.

The effect of error bars produces a broadening effect on the observed distribution.
In the case where all points have the same error we end up with a gaussian convolution of a gaussian which is itself another gaussian with the variances added in quadrature.
In the case where the errors themselves have a (likely unknown) distribution the normally distributed data will appear non-normal (often with broad wings).

Taking the mean is likely unaffected by the convoltion of errors since points will, on average, be raise higher/lower at an equal rate. The standard deviation should always increase due to the broadening effect of the errors.
The problem therefore is how to recover the parameters of the underlying distribution.

This is the problem we tackle in this notebook using statistical deconvolution. This approach uses a likelihood function, accounting for each individual measurement error, to recover the parameters.


## Imports

In [ ]:
import numpy as np  # import numpy for constructing mock data and initial conditions
import essrs        # import rust-based sampler machinery
import time

## Define parameters

In [ ]:
N_DATA_POINTS: int = 10000  # the number of mock data points (with errors) to fit
N_PARAMETERS: int = 2       # the number of parameters in the fit
N_WALKERS: int = 100        # the number of walkers running in the ensemble

# we shall draw points from this distribution
TRUE_MEAN: float = 5.0      # true mean of underlying distribution
TRUE_SIGMA: float = 1.5     # true standard deviation of the underlying distribution

# this defines the convolution distribution
# because we are dispersing the errors we cannot simply take naive mean and subtract error in quadrature
ERROR_CENTROID: float = 1.2 # the mean error which will be applied to points
ERROR_SIGMA: float = 0.1    # standard deviation around mean of error which is applied to points

# MCMC specific constants
N_BURN_IN: int = 500
N_EVAL: int = 1000

## Generate mock data

The print statements at the bottom highlight the point made earlier.
The mean is only minorly different after convoltion (and the difference may even be a sampling effect from sampling the points).
The standard deviation however is significantly larger highlighting the issue with taking naive estimates of the parameters.

In [ ]:
# draw the underlying data points and their errors
data = np.random.normal(TRUE_MEAN, TRUE_SIGMA, N_DATA_POINTS)
error = np.random.normal(ERROR_CENTROID, ERROR_SIGMA, N_DATA_POINTS)

print("True estimates of population parameters:")
print(f"MEAN  = {np.mean(data):5.3f}")
print(f"SIGMA = {np.std(data):5.3f}\n\n")

# apply a convolution by shifting each data point from true value by sampling from error
convolution = error * np.random.normal(0.0, 1.0, N_DATA_POINTS)
conv_data = data + convolution

# construct the data to be fitted
# NOTE: we are using convolved data so the model has no knowledge of the 'true' data points
fit_data = np.vstack((conv_data, error)).T

print("Naive estimates of global parameters:")
print(f"{np.mean(conv_data):5.3f} ~= {TRUE_MEAN:3.1f}")
print(f"{np.std(conv_data):5.3f} > {TRUE_SIGMA:3.1f}")

## Generate initial conditions

The naive estimates are good candidates for initial conditions of the fit.
The initial conditions are generated by taking the mean and sigma values and randomly distributing points around those values.

In [ ]:
ics = (
    np.array([np.mean(data), np.std(data)]) *
    np.random.normal(1, 0.05, size=(N_WALKERS, N_PARAMETERS))
)

## MCMC fit

In [ ]:
start = time.perf_counter()

# we use the normal_1d_deconv function which handles the inherent error in the points
fit = essrs.normal_1d.normal_1d_deconv(
    max_n_steps=N_EVAL,        # max number of steps to run the sampler
    n_burn_in=N_BURN_IN,          # number of burn in steps to discard prior to main sampling stage
    initial_conditions=ics, # initial conditions
    data=fit_data           # data, with errors, that must be fit
)

elapsed = time.perf_counter() - start
print(f"MCMC took {elapsed:.3f} seconds")

## Results

You can see from the results below that, on top of deriving associated uncertainties in our parameter estimates, the estimate of the population mean is now once again much closer to the true value.
This demonstrates the power of statistical deconvolution and the utility of the `essrs` package. 

In [ ]:
print("Statistical deconvolution estimates of global parameters:")
print(f"{np.mean(fit[-1, :, 0].ravel()):5.3f} +- {np.std(fit[-1, :, 0].ravel()):6.4f} | ~= {TRUE_MEAN:3.1f}")
print(f"{np.mean(fit[-1, :, 1].ravel()):5.3f} +- {np.std(fit[-1, :, 1].ravel()):6.4f} | ~= {TRUE_SIGMA:3.1f}")

## Benchmarking Against Python Implementations

In [ ]:
import emcee

In [ ]:
def log_probability(
        theta: np.ndarray,
        data: np.ndarray) -> float:

    """Function implementing the equivalent log-likelihood in python"""
    # compute the sum in quadrature
    sigma_quad_sqr = data[:, 1]**2 + theta[1]**2

    logl = -0.5 * np.log(2 * np.pi * sigma_quad_sqr) -0.5 * (theta[0] - data[:,0])**2 / sigma_quad_sqr

    return np.sum(logl)


sampler = emcee.EnsembleSampler(
    N_WALKERS,
    N_PARAMETERS,
    log_probability,
    args=(fit_data,),
    moves=emcee.moves.DEMove(),
)

In [ ]:
start = time.perf_counter()

state = sampler.run_mcmc(ics, N_BURN_IN)
# Reset so burn-in isn't included in the chain
sampler.reset()
sampler.run_mcmc(state, N_EVAL)

elapsed = time.perf_counter() - start
print(f"MCMC took {elapsed:.3f} seconds")

In [ ]:
chains = sampler.get_chain()
print(np.shape(chains))

print("Statistical deconvolution estimates of global parameters -- EMCEE:")
print(f"{np.mean(chains[-1, :, 0].ravel()):5.3f} +- {np.std(chains[-1, :, 0].ravel()):6.4f} | ~= {TRUE_MEAN:3.1f}")
print(f"{np.mean(chains[-1, :, 1].ravel()):5.3f} +- {np.std(chains[-1, :, 1].ravel()):6.4f} | ~= {TRUE_SIGMA:3.1f}")
